# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
import re
import tensorflow as tf
from models.llama3.generation import Llama
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [ ]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset3.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)

# API Setup

In [ ]:
# Sonar
 
api_key = os.environ.get("PERPLEXITY_API_KEY")

if api_key:
    print('successful')

url = "https://api.perplexity.ai/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

In [ ]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [ ]:
# DeepSeek

from openai import OpenAI
api_key = os.environ.get("DEEPSEEK_API_KEY")

if api_key:
    print('successful')

client = OpenAI(api_key="<DeepSeek API Key>", base_url="https://api.deepseek.com")

# Zero-Shot

## SONAR

In [ ]:
# Test

query = 'How many total disk access is needed to search a record using two level indexing?'
payload = {
    "model": "sonar",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]
         
         query : {query}"""}
    ],
    "max_tokens": 100,
    "temperature": 0.5
}
response = requests.post(url, headers=headers, json=payload).json()
reply = response["choices"][0]["message"]["content"]

payload = {
    "model": "sonar",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"""Extract the blooms level from my previous reponse. Answer only in one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]
         previous response : {reply}"""}
    ],
    "max_tokens": 100,
    "temperature": 0.5
}
response = requests.post(url, headers=headers, json=payload).json()
reply = response["choices"][0]["message"]["content"]

print(reply)


### Assign labels

In [ ]:
pred_labels= []

for query in tqdm(queries):
    payload = {
        "model": "sonar",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
                [knowledge, comprehension, application, analysis, synthesis, evaluation]
            
            query : {query}"""}
        ],
        "max_tokens": 100,
        "temperature": 0.5
    }

    response = requests.post(url, headers=headers, json=payload).json()
    reply = response["choices"][0]["message"]["content"]

    if reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:

        payload = {
            "model": "sonar",
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": f"""Extract the blooms level from my previous reponse. Answer only in one word from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                previous response : {reply}"""}
            ],
            "max_tokens": 100,
            "temperature": 0.5
        }
        response = requests.post(url, headers=headers, json=payload).json()

    reply = response["choices"][0]["message"]["content"]

    pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

## DeepSeek

## LLAMA4-Scout

In [ ]:
# Test

query = 'How many total disk access is needed to search a record using two level indexing?'
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
                [knowledge, comprehension, application, analysis, synthesis, evaluation]
            
            query : {query}""",
        }
    ],
    model="meta-llama/llama-4-scout-17b-16e-instruct",
)

reply = chat_completion.choices[0].message.content.lower()
print(reply)

while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Extract the blooms level from my previous reponse. Answer only in one word without punctuation from: 
                    [knowledge , comprehension , application , analysis, synthesis , evaluation]
                previous response : {reply}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()
    print(reply)


print(reply)

### Assign Labels

In [ ]:
pred_labels = []

for query in tqdm(queries):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from my previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    pred_labels.append(reply.lower())

## LLAMA4-Madvick

### Assign Labels

In [ ]:
pred_labels = []

for query in tqdm(queries):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from my previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-maverick-17b-128e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    pred_labels.append(reply.lower())

print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))